# Transformer Decoder & Sampling (How LLMs Generate Text)

In the previous notebooks we covered:
- **Tokenization** – how text becomes integers
- **Embeddings** – how integers become vectors
- **Attention / Encoder block** – how tokens talk to each other

In this notebook we focus on **how LLMs actually *generate* text**:
- The **Transformer decoder** and **causal masking**
- **Encoder–decoder vs decoder-only** architectures
- How we go from **logits → probabilities → tokens**
- **Greedy decoding** vs **sampling**
- **Temperature, top-k, top-p**

We’ll keep code minimal and use it mainly to *visualize* what’s going on.

## 1. Encoder–Decoder vs Decoder-Only

The original Transformer paper ("Attention is All You Need") used an **encoder–decoder** architecture for **machine translation**:

```text
Input sentence (source language)  →  Encoder  →  Encoded representations
                                                   ↓
                                          Decoder  →  Output sentence (target language)
```

- The **encoder** reads the whole input sentence and produces hidden representations.
- The **decoder** then generates the output, one token at a time, using:
  - The past generated tokens (through **self-attention** with causal mask)
  - The encoder outputs (through **cross-attention**)

For **LLMs like GPT**, we usually use a **decoder-only** architecture:

```text
Tokens so far  →  Decoder-only Transformer  →  Next-token distribution
```

Here, there is **no separate encoder**. The same stack of decoder blocks processes the prompt and then keeps generating tokens autoregressively.

## 2. What Makes a Decoder Block Different?

A **decoder block** has (in the full encoder–decoder architecture):

1. **Masked self-attention** over previous output tokens (cannot see future tokens)
2. **Cross-attention** over encoder outputs (to look at the input sentence)
3. **Feed-forward network (FFN)**

Rough structure:

```text
           ┌────────────────────┐
x  ───────▶│ Masked Self-Attn   │
           └────────────────────┘
                      │ + x (residual)
                      ▼
                 LayerNorm
                      │
           ┌────────────────────┐
           │ Cross-Attention    │  (attend to encoder outputs)
           └────────────────────┘
                      │ + (prev)
                      ▼
                 LayerNorm
                      │
           ┌────────────────────┐
           │ Feed-Forward (FFN) │
           └────────────────────┘
                      │ + (prev)
                      ▼
                 LayerNorm
```

In **decoder-only LLMs**, we usually **drop the cross-attention** and just stack masked self-attention + FFN blocks.

The key new idea here compared to the encoder is the **causal mask**.

## 3. Causal Mask: "No Peeking into the Future"

During **generation**, token at position **t** must only use tokens at positions ≤ t.

We enforce this with a **causal (triangular) mask** that blocks attention to future positions.

For a sequence of length 4, the mask (1 = allowed, 0 = blocked) looks like:

```text
      keys →   1   2   3   4
queries ↓
   1        [ 1   0   0   0 ]
   2        [ 1   1   0   0 ]
   3        [ 1   1   1   0 ]
   4        [ 1   1   1   1 ]
```

- Row = **query position** (current token)
- Column = **key position** (token we want to look at)
- Above the diagonal are **future tokens** → they get masked (blocked)

In practice, we add **-∞ (or a very large negative number)** to the attention scores where the mask is 0, so after softmax those positions get probability ~0.

In [ ]:
import torch

def causal_mask(seq_len: int):
    """Return a [seq_len, seq_len] causal mask with 0 for allowed, -inf for blocked."""
    # torch.triu creates an upper-triangular matrix. We use it to find future positions.
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)  # 1 above diagonal, 0 else
    # Convert 1 → -inf (blocked), 0 → 0 (keep)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

mask = causal_mask(4)
mask

This mask is **added to the attention scores** before softmax:

```python
scores = Q @ K.T / sqrt(d_k)
scores = scores + mask  # future positions get -inf
attn_weights = softmax(scores, dim=-1)
```

This guarantees **strict left-to-right generation**.

## 4. From Logits to Probabilities

After the final decoder layer, we usually have a **linear layer** that maps hidden states to **logits over the vocabulary**:

```text
hidden state at position t  →  linear layer  →  logits (one value per token in vocab)
```

- **Logits** are just raw scores (any real numbers).
- To turn logits into **probabilities**, we apply **softmax**:

```text
p_i = exp(logit_i) / sum_j exp(logit_j)
```

Then we can:
- **Pick the most likely token** (greedy), or
- **Sample** from this distribution (with temperature, top-k, top-p, etc.).

In [ ]:
import torch
import torch.nn.functional as F

# Suppose we have logits for 5 fake tokens
logits = torch.tensor([2.0, 1.0, 0.5, -1.0, 3.0])
probs = F.softmax(logits, dim=-1)

print("Logits:", logits)
print("Probabilities:", probs)
print("Sum of probabilities:", probs.sum())

## 5. Greedy Decoding (Deterministic)

The simplest decoding strategy:

1. Compute probabilities for the next token.
2. Pick the **argmax** (the token with highest probability).
3. Append it to the sequence.
4. Repeat.

This is called **greedy decoding**. It is:
- **Deterministic** (same prompt → always same output)
- Often **reasonable**, but can be **boring** and can get stuck in loops or repetitive patterns.

To make outputs more diverse and controllable, we use **sampling methods**.

## 6. Temperature: Controlling Sharpness

We adjust logits before softmax using a **temperature** parameter **T > 0**:

```text
probs_i = softmax( logits_i / T )
```

- **T = 1.0** → original distribution
- **T < 1.0** (e.g. 0.7) → distribution becomes **sharper**
  - Model becomes more **confident / deterministic**
- **T > 1.0** (e.g. 1.5) → distribution becomes **flatter**
  - Model explores **more diverse** tokens

Intuition: temperature controls how much we **trust** the model’s top choice vs exploring other options.

In [ ]:
def softmax_with_temperature(logits, T=1.0):
    return F.softmax(logits / T, dim=-1)

for T in [0.5, 1.0, 2.0]:
    probs_T = softmax_with_temperature(logits, T)
    print(f"Temperature = {T}")
    print(probs_T, "\n")

You can inspect how higher temperature spreads probability mass across more tokens, while lower temperature concentrates it on the top ones.

## 7. Top-k Sampling

**Top-k** sampling keeps only the **k most likely tokens** and sets the rest to probability 0.

Steps:
1. Sort tokens by logit/probability.
2. Keep the top **k** tokens.
3. Renormalize probabilities over this smaller set.
4. Sample from that.

Effect:
- Limits generation to a **small but reasonable** set of tokens.
- Avoids choosing extremely low-probability tokens (which might be nonsense).

Example: if **k = 5** and vocabulary has 50,000 tokens, we only sample from the best 5 at each step.

In [ ]:
def top_k_logits(logits, k):
    """Keep only top-k logits, set the rest to -inf."""
    values, indices = torch.topk(logits, k)
    mask = torch.ones_like(logits, dtype=torch.bool)
    mask[indices] = False  # these are the top-k positions, keep them
    filtered_logits = logits.clone()
    filtered_logits[mask] = float('-inf')
    return filtered_logits

k = 2
filtered = top_k_logits(logits, k)
probs_top_k = F.softmax(filtered, dim=-1)

print("Original probs:", F.softmax(logits, dim=-1))
print(f"\nTop-{k} filtered probs:", probs_top_k)

Notice how after top-k filtering, only **k positions** have non-zero probability.

## 8. Top-p (Nucleus) Sampling

**Top-p**, or **nucleus sampling**, keeps the **smallest set of tokens whose cumulative probability ≥ p**.

Steps:
1. Sort tokens by probability (descending).
2. Take tokens until their **cumulative probability** reaches some threshold **p** (e.g. 0.9).
3. Renormalize probabilities on this dynamic set.
4. Sample from that.

Effect:
- The number of tokens considered is **dynamic** (can be 3 in one step, 10 in another).
- It adapts to the **shape** of the distribution.
- Often produces **more natural** text than fixed top-k.

You can think of p as: "I want to sample from the **top 90% probability mass** of the distribution."

In [ ]:
def top_p_logits(logits, p=0.9):
    """Keep the smallest set of tokens with cumulative prob >= p."""
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)

    # Find how many tokens we need to reach p
    mask = cumulative > p
    # First position where cumulative exceeds p
    cutoff_index = torch.nonzero(mask, as_tuple=False)
    if len(cutoff_index) > 0:
        cutoff = cutoff_index[0].item() + 1  # keep up to this index (inclusive)
    else:
        cutoff = len(logits)

    # Everything after cutoff is set to -inf
    keep_indices = sorted_indices[:cutoff]
    filtered_logits = logits.clone()
    keep_mask = torch.zeros_like(logits, dtype=torch.bool)
    keep_mask[keep_indices] = True
    filtered_logits[~keep_mask] = float('-inf')
    return filtered_logits

filtered_p = top_p_logits(logits, p=0.9)
probs_top_p = F.softmax(filtered_p, dim=-1)

print("Original probs:", F.softmax(logits, dim=-1))
print("\nTop-p filtered probs:", probs_top_p)
print("Sum of probs after top-p:", probs_top_p.sum())

You can experiment with different **p** values (e.g. 0.8, 0.9, 0.95) and see how many tokens are kept.

## 9. Tiny Autoregressive Generation Demo (Toy)

Now let’s write a **very simplified generation loop** that:
- Pretends we have some model that returns random logits
- Applies **temperature**
- Optionally applies **top-k** or **top-p**
- Samples the next token

This is not a real language model – just a **toy demo** to illustrate the decoding pipeline.

Pipeline for each step:

```text
current_tokens
   ↓
fake_model → logits
   ↓
optional: divide by T
   ↓
optional: top-k / top-p filter
   ↓
softmax → probabilities
   ↓
sample one token
   ↓
append to sequence
```

In [ ]:
import random

def fake_model_logits(vocab_size):
    """Return random logits (just for demo)."""
    return torch.randn(vocab_size)

def sample_next_token(logits, temperature=1.0, k=None, p=None):
    # Apply temperature
    logits = logits / temperature

    # Apply top-k
    if k is not None:
        logits = top_k_logits(logits, k)

    # Apply top-p
    if p is not None:
        logits = top_p_logits(logits, p)

    probs = F.softmax(logits, dim=-1)
    # Sample one token index using the probabilities
    next_token = torch.multinomial(probs, num_samples=1).item()
    return next_token, probs

def generate_toy_sequence(vocab_size=10, max_steps=10, temperature=1.0, k=None, p=None):
    tokens = []
    for step in range(max_steps):
        logits = fake_model_logits(vocab_size)
        token, probs = sample_next_token(logits, temperature, k, p)
        tokens.append(token)
    return tokens

print("Greedy-ish low temperature (T=0.5), no top-k/p:")
print(generate_toy_sequence(temperature=0.5))

print("\nHigher temperature (T=2.0), more randomness:")
print(generate_toy_sequence(temperature=2.0))

print("\nTop-k=3, T=1.0:")
print(generate_toy_sequence(temperature=1.0, k=3))

print("\nTop-p=0.8, T=1.0:")
print(generate_toy_sequence(temperature=1.0, p=0.8))

In a **real LLM**, this loop would:
- Feed the **current sequence** into the model
- Use **KV cache** so we don’t recompute everything each time
- Return logits for the **next token**
- Then apply temperature / top-k / top-p as above

But the overall decoding **logic** is the same as our toy example.

## 10. Summary

In this notebook we saw:

- How the **decoder** differs from the encoder, especially with **causal masking** and **autoregressive** behavior.
- The difference between **encoder–decoder** and **decoder-only** architectures.
- How **hidden states → logits → probabilities** using a final linear layer and softmax.
- **Greedy decoding** vs **sampling-based decoding**.
- How **temperature** controls sharpness of the distribution.
- How **top-k** and **top-p** sampling restrict the set of tokens we sample from.

This completes the story of **how a Transformer not only understands, but also *generates* text**.